# Whisper Transcription Backend
Run all cells top to bottom. Cell 4 will print your ngrok URL — paste it into the frontend.

In [ ]:
# Cell 1 — Install dependencies
!pip install faster-whisper flask flask-cors pyngrok -q

In [ ]:
# Cell 2 — Set your ngrok auth token
# Get a free token at https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "PASTE_YOUR_TOKEN_HERE"

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)

In [ ]:
# Cell 3 — Flask app with SSE streaming
import os, uuid, json, queue, threading
from flask import Flask, request, Response, jsonify
from flask_cors import CORS
from faster_whisper import WhisperModel

app = Flask(__name__)
CORS(app)

UPLOAD_DIR = "/tmp/whisper_uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)

# job_id -> { status, progress, transcript, error }
jobs = {}

print("Loading Whisper turbo model...")
model = WhisperModel("turbo", device="cuda", compute_type="float16")
print("Model loaded.")


def run_transcription(job_id, filepath):
    try:
        jobs[job_id]["status"] = "transcribing"
        segments, info = model.transcribe(filepath, beam_size=5)

        jobs[job_id]["language"] = info.language
        transcript_lines = []

        # We need to consume the generator to get total — estimate from duration
        duration = info.duration

        for segment in segments:
            line = f"[{segment.start:.2f}s --> {segment.end:.2f}s] {segment.text.strip()}"
            transcript_lines.append(line)
            progress = min(int((segment.end / duration) * 100), 99) if duration else 0
            jobs[job_id]["progress"] = progress
            jobs[job_id]["latest_line"] = line

        jobs[job_id]["transcript"] = "\n".join(transcript_lines)
        jobs[job_id]["progress"] = 100
        jobs[job_id]["status"] = "done"

    except Exception as e:
        jobs[job_id]["status"] = "error"
        jobs[job_id]["error"] = str(e)
    finally:
        if os.path.exists(filepath):
            os.remove(filepath)


@app.route("/upload", methods=["POST"])
def upload():
    if "file" not in request.files:
        return jsonify({"error": "No file provided"}), 400

    f = request.files["file"]
    job_id = str(uuid.uuid4())
    filepath = os.path.join(UPLOAD_DIR, f"{job_id}_{f.filename}")
    f.save(filepath)

    jobs[job_id] = {
        "status": "queued",
        "progress": 0,
        "transcript": "",
        "latest_line": "",
        "language": "",
        "error": ""
    }

    thread = threading.Thread(target=run_transcription, args=(job_id, filepath))
    thread.daemon = True
    thread.start()

    return jsonify({"job_id": job_id})


@app.route("/stream/<job_id>")
def stream(job_id):
    def event_generator():
        import time
        last_line = ""
        while True:
            job = jobs.get(job_id)
            if not job:
                yield f"data: {json.dumps({'error': 'Job not found'})}\n\n"
                break

            payload = {
                "status": job["status"],
                "progress": job["progress"],
                "language": job["language"],
                "latest_line": job["latest_line"] if job["latest_line"] != last_line else "",
            }

            if job["latest_line"] != last_line:
                last_line = job["latest_line"]

            yield f"data: {json.dumps(payload)}\n\n"

            if job["status"] in ("done", "error"):
                break

            time.sleep(1)

    return Response(event_generator(), mimetype="text/event-stream",
                    headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})


@app.route("/result/<job_id>")
def result(job_id):
    job = jobs.get(job_id)
    if not job:
        return jsonify({"error": "Job not found"}), 404
    return jsonify({"transcript": job["transcript"], "language": job["language"]})


@app.route("/health")
def health():
    return jsonify({"ok": True})


print("Flask app ready.")

In [ ]:
# Cell 4 — Start server and print your URL
from pyngrok import ngrok
import threading

public_url = ngrok.connect(5000).public_url
print("\n" + "="*50)
print(f"BACKEND URL: {public_url}")
print("Paste this into the frontend HTML file")
print("="*50 + "\n")

threading.Thread(target=lambda: app.run(port=5000, threaded=True)).start()